<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/11_parameter_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0 - Best Practice)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# -----------------------------------------------------------------
# 1. Google Drive 마운트
# -----------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# -----------------------------------------------------------------
# 2. GitHub 최신화 및 경로 설정 (충돌 없는 강제 동기화)
# -----------------------------------------------------------------
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

# 핵심: 작업 디렉토리를 프로젝트 루트로 완벽히 고정하여 requirements.txt를 찾게 함
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# -----------------------------------------------------------------
# 3. 커스텀 모듈(src) 실행을 통한 의존성 설치 및 데이터셋 로드
# -----------------------------------------------------------------
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # env_setup.py 내부의 함수를 호출하여 requirements.txt 기반 설치 실행
    # (이 단계에서 torchcrepe, pretty_midi 등이 정상 설치됩니다)
    init_colab_env()

    # 데이터셋 복사
    MY_DRIVE_DATA_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_DATA_PATH, force_update=False)

except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")
    print("   (src/utils.py의 'import shutil' 오타가 수정되었는지 확인하세요!)")
except Exception as e:
    print(f"❌ 셋업 중단: {e}")

# -----------------------------------------------------------------
# 4. 글로벌 라이브러리 사전 적재
# -----------------------------------------------------------------
import librosa
import numpy as np
import pandas as pd
import torchcrepe
import pretty_midi

print("\n🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.")

🚀 Bass Separator 통합 환경 설정을 시작합니다...
Mounted at /content/drive
📦 레포지토리 클론 중... (Bass-separator)
🚀 환경 설정을 시작합니다...

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 확인 중...
📄 requirements.txt 파일을 발견했습니다. 의존성 패키지를 설치합니다...
📦 패키지 일괄 설치 진행 중...
✅ 패키지 일괄 설치 완료.

🏥 설치 무결성 점검 (Health Check)...
✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!

🎉 모든 환경 설정이 완료되었습니다!
🚀 데이터 복사 시작...
   📂 Source: /content/drive/MyDrive/Bass_separator/dataset
   📂 Dest  : ./dataset
🎉 데이터 준비 완료! (총 5개 파일 복사됨)

🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.


In [19]:
# ==================================================================
# [Phase 6] Fast Tuning Test Harness (Demucs Bypassed)
# ==================================================================
import librosa
import numpy as np
import pandas as pd
import torch
import torchcrepe
import pretty_midi
import scipy.signal
from dataclasses import dataclass

# -----------------------------------------------------------------
# 1. Data Structure
# -----------------------------------------------------------------
@dataclass(frozen=True)
class NoteEvent:
    time: float
    duration: float
    midi_note: int
    confidence: float

# -----------------------------------------------------------------
# 2. Tracking Module (Type A Error Tuning Target)
# -----------------------------------------------------------------
# [복구] 슬랩 파트 옥타브 튀김을 방지하는 스마트 보정 모듈
def clean_octave_errors_smart(f0_array, onset_mask, window_size=7, onset_tolerance=2):
    f0_clean = f0_array.copy()
    mask = (f0_clean > 0) & (~np.isnan(f0_clean))
    if np.sum(mask) == 0: return f0_clean

    midi_notes = np.zeros_like(f0_clean)
    midi_notes[mask] = librosa.hz_to_midi(f0_clean[mask])

    midi_series = pd.Series(midi_notes)
    trend = midi_series.where(mask).rolling(window=window_size, center=True, min_periods=1).median().values

    indices = np.where(mask)[0]
    for i in indices:
        if np.isnan(trend[i]): continue
        diff = midi_notes[i] - trend[i]
        is_octave_jump = (10 <= diff <= 14) or (22 <= diff <= 26) or (-14 <= diff <= -10)

        if is_octave_jump:
            start_idx = max(0, i - onset_tolerance)
            end_idx = min(len(onset_mask), i + onset_tolerance + 1)
            is_intentional_attack = np.any(onset_mask[start_idx:end_idx])

            if not is_intentional_attack:
                if 10 <= diff <= 14: midi_notes[i] -= 12
                elif 22 <= diff <= 26: midi_notes[i] -= 24
                elif -14 <= diff <= -10: midi_notes[i] += 12

    f0_clean[mask] = librosa.midi_to_hz(midi_notes[mask])
    return f0_clean

def get_f0_crepe_robust_tuned(audio, sr, onset_mask, hop_length=160, fmax=500):
    """
    [Tuning Target 1] CREPE Pitch Tracking & Dynamic Thresholding
    """
    sos = scipy.signal.butter(4, 35, 'hp', fs=sr, output='sos')
    audio = scipy.signal.sosfilt(sos, audio).astype(np.float32)
    audio = librosa.util.normalize(audio)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 추론 (메모리 문제 방지를 위해 chunking 로직 축약 - 단일 테스트용)
    audio_tensor = torch.tensor(audio).unsqueeze(0).to(device)
    f0, confidence = torchcrepe.predict(
        audio_tensor, sr, hop_length=hop_length, fmin=40, fmax=fmax,
        model='tiny', decoder=torchcrepe.decode.argmax,
        return_periodicity=True, device=device
    )

    f0 = f0.squeeze().cpu().numpy()
    confidence = confidence.squeeze().cpu().numpy()

    # [신설] 옥타브 에러 평탄화 (슬랩 튀는 음 억제)
    f0 = clean_octave_errors_smart(f0, onset_mask)

    # ---------------------------------------------------------
    # 🔴 [HYPERPARAMETERS TO TUNE] 🔴
    # ---------------------------------------------------------
    # [Tuning 3-1] High-Freq 관용도 약간 상향 (0.7 -> 0.6)
    mask_low = (f0 < 80) & (confidence < 0.25)   # 저음 서스테인은 그대로 보존
    mask_mid = (f0 >= 80) & (f0 <= 200) & (confidence < 0.4)
    mask_high = (f0 > 200) & (confidence < 0.6)  # 하이 노트 인식률 복구

    f0[mask_low | mask_mid | mask_high] = np.nan
    f0[f0 > fmax] = np.nan

    # median filter 삭제
    return f0, confidence

# -----------------------------------------------------------------
# 3. Parsing Module (Type B Error Tuning Target)
# -----------------------------------------------------------------
def parse_f0_to_events_tuned(f0_array, confidence_array, onset_mask, hop_length=160, sr=16000):
    """
    [Tuning Target 2] Debouncing & Note Extraction
    """
    events = []
    frame_time = hop_length / sr
    current_note = None
    note_start_frame = 0
    blank_counter = 0

    # ---------------------------------------------------------
    # 🔴 [HYPERPARAMETERS TO TUNE] 🔴
    # ---------------------------------------------------------
    # 1. 최소 유지 프레임 (ex. 5 = 50ms). 이보다 짧은 음은 노이즈로 간주.
    MIN_DURATION_FRAMES = 6

    # 2. 결측치 관용도 (5 = 50ms). 이 프레임 내의 NaN은 같은 음표로 이어붙임.
    TOLERANCE_FRAMES = 5
    # ---------------------------------------------------------
    # [신설] Latency Compensation Value (초 단위)
    # 이미지 2의 Latency를 시각적으로 보정하기 위한 고정값 (예: 15ms = 0.015)
    LATENCY_COMP_SEC = 0.010
    # ---------------------------------------------------------
    # [신설] Same-Pitch Retriggering용 Confidence Threshold
    # 피치는 같지만 Onset이 탐지 안 될 때, 해당 프레임의 신뢰도가 이보다 낮으면 Retriggering 시도
    # 서스테인 보호를 위해 임계값을 더 엄격하게(낮게) 조정
    RETRIGGER_CONF_THRESH = 0.2
    # ---------------------------------------------------------

    valid_mask = (f0_array > 0) & (~np.isnan(f0_array))
    midi_array = np.full(len(f0_array), np.nan)
    # Hz -> MIDI Discretization (Round)
    midi_array[valid_mask] = np.round(librosa.hz_to_midi(f0_array[valid_mask]))

    for i, midi_val in enumerate(midi_array):
        current_onset = onset_mask[i]

        if not np.isnan(midi_val):
            midi_note = int(midi_val)
            conf_val = confidence_array[i] # 현재 프레임의 모델 신뢰도

            blank_counter = 0
            if current_note is None:
                current_note = midi_note
                note_start_frame = i

            # 🔴 [교정] 일반적인 분할 (피치 변경 또는 확실한 타격)
            elif (current_note != midi_note) or (current_onset is True):
                duration_frames = i - note_start_frame
                if duration_frames >= MIN_DURATION_FRAMES:
                    conf_avg = float(np.mean(confidence_array[note_start_frame:i]))
                    comp_start_time = max(0, (note_start_frame * frame_time) - LATENCY_COMP_SEC)
                    events.append(NoteEvent(comp_start_time, duration_frames * frame_time, current_note, conf_avg))
                current_note = midi_note
                note_start_frame = i

            # 🔴 [교정] Confidence 딥(Dip) 발생 시 -> 노트 종료 후 Blank 상태로 진입
            elif (conf_val < RETRIGGER_CONF_THRESH):
                duration_frames = i - note_start_frame
                if duration_frames >= MIN_DURATION_FRAMES:
                    conf_avg = float(np.mean(confidence_array[note_start_frame:i]))
                    comp_start_time = max(0, (note_start_frame * frame_time) - LATENCY_COMP_SEC)
                    events.append(NoteEvent(comp_start_time, duration_frames * frame_time, current_note, conf_avg))
                # [핵심 버그 수정] 노이즈 구간이므로 새 노트를 시작하지 않고 대기 상태로 전환
                current_note = None

    if current_note is not None:
        duration_frames = len(midi_array) - note_start_frame
        if duration_frames >= MIN_DURATION_FRAMES:
            conf = float(np.mean(confidence_array[note_start_frame:]))

            # [지연보정] Heuristic Latency Compensation 적용
            comp_start_time = max(0, (note_start_frame * frame_time) - LATENCY_COMP_SEC)

            events.append(NoteEvent(comp_start_time, duration_frames * frame_time, current_note, conf))

    return events

# -----------------------------------------------------------------
# 4. MIDI Export & Runner (Updated with Sensitive Onset Detection)
# -----------------------------------------------------------------
def run_tuning_test(audio_path):
    print(f"🎵 오디오 로딩 중: {audio_path}")
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    hop_length = 160

    # 1. Onset 마스크 먼저 생성
    print("🥁 Onset (어택) 탐지 및 마스크 생성 중...")
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length, fmax=400)
    onset_frames = librosa.onset.onset_detect(
        onset_envelope=onset_env, sr=sr, hop_length=hop_length,
        wait=3, pre_avg=3, post_avg=3, delta=0.07
    )
    # 추정 프레임 길이 연산
    estimated_frames = int(np.ceil(len(y) / hop_length))
    onset_mask = np.zeros(estimated_frames, dtype=bool)
    valid_onsets = onset_frames[onset_frames < estimated_frames]
    onset_mask[valid_onsets] = True

    # 2. CREPE 추론 (onset_mask 주입)
    print("🧠 피치 트래킹 수행 중 (CREPE + Octave Correction)...")
    f0, conf = get_f0_crepe_robust_tuned(y, sr, onset_mask=onset_mask, hop_length=hop_length)

    # 3. 파싱
    print("⚙️ 이벤트 파싱 및 디바운싱 수행 중...")
    events = parse_f0_to_events_tuned(f0, conf, onset_mask=onset_mask, hop_length=hop_length, sr=sr)

    out_midi_path = "/content/tuned_test_result.mid"
    events_to_midi(events, out_midi_path)
    return out_midi_path

In [20]:
# 🚀 실행부
import shutil
import os

TEST_AUDIO_PATH = "/content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav"
result_midi = run_tuning_test(TEST_AUDIO_PATH)
source_path = result_midi

# Destination path in Google Drive
destination_dir = "/content/drive/MyDrive/Bass_separator/"
destination_file_name = "midi_tunning_test7.mid"
destination_path = destination_dir + destination_file_name

# Ensure the destination directory exists
os.makedirs(destination_dir, exist_ok=True)

# Copy the file
try:
    shutil.copy(source_path, destination_path)
    print(f"✅ MIDI 파일이 Google Drive에 성공적으로 저장되었습니다: {destination_path}")
except Exception as e:
    print(f"❌ MIDI 파일 저장 중 오류 발생: {e}")

🎵 오디오 로딩 중: /content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav
🥁 Onset (어택) 탐지 및 마스크 생성 중...


/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


🧠 피치 트래킹 수행 중 (CREPE + Octave Correction)...
⚙️ 이벤트 파싱 및 디바운싱 수행 중...
✅ MIDI 추출 완료: /content/tuned_test_result.mid (총 201개 노트)
✅ MIDI 파일이 Google Drive에 성공적으로 저장되었습니다: /content/drive/MyDrive/Bass_separator/midi_tunning_test7.mid


### 6.2. Parameter Tuning Iterations (Phase 6)

본 섹션은 분리된 베이스 음원(Stem)을 대상으로 한 피치 트래킹 및 파싱 모듈의 하이퍼파라미터 최적화 실험 기록입니다. E2E 파이프라인 정합성을 위해, 정량적 수치 변화에 따른 오디오-MIDI 시각적 대조(Time-alignment) 결과를 기반으로 튜닝을 진행했습니다.

---

#### 🧪 Iteration 1: Baseline Test (`midi_tunning_test.mid`)

**[Parameters]**
* **Tracking Module:**
  * `BASE_CONF_THRESH`: `0.4` (저음역대 포함 전체 적용)
  * `HIGH_FREQ_HZ`: `200.0` / `HIGH_FREQ_CONF_THRESH`: `0.7` (타악기 블리딩 및 배음 에러 방어선)
* **Parsing Module:**
  * `MIN_DURATION_FRAMES`: `5` (50ms 미만 노이즈 필터링)
  * `TOLERANCE_FRAMES`: `5` (50ms 결측치 브릿징)

**[Results]**
1. **Low-end Early Cutoff:** 4번 줄의 낮은 베이스 노트 인식률이 현저히 떨어지며, 실제 연주보다 음이 조기에 끊어짐(Sustain loss).
2. **High-end Stability:** 비교적 높은 음역대의 라인은 빠른 속주에서도 양호한 인식률을 보임.
3. **Transient Artifacts (Double-triggering):** 슬랩의 썸(Thumb) 주법은 피치를 잘 추적하나, 플럭(Pluck) 주법의 강한 타격 구간에서 발생하는 비화성(Inharmonic) 배음이 독립적인 짧은 노트로 튀는 현상 발생.
4. **Slide Fragmentation:** 슬라이드 주법 시 연속적인 주파수(Hz)를 이산적인 MIDI 노트로 변환하는 과정에서, 경유하는 반음들이 짧고 급격한 속주로 쪼개지는 현상 발생.

**[Next Steps (Action Item)]**
* 저음역대 인식률 복구를 위해 주파수 대역별 동적 임계값 도입 (저역대 Threshold 하향).
* 플럭 주법의 타격 노이즈(Pitch Spike)를 평탄화하기 위해 F0 배열에 `Median Filtering` 도입.
* 슬라이드 파편화 방지를 위해 `MIN_DURATION_FRAMES` 상향 조정.

---

#### 🧪 Iteration 2: Aggressive Filtering (`midi_tunning_test2.mid`)

**[Parameters]**
* **Tracking Module (3-Tier Dynamic Threshold & Median Filter):**
  * Low-Freq Gate (`< 80Hz`): `0.25` (4번 줄 조기 끊김 방지)
  * Mid-Freq Gate (`80 ~ 200Hz`): `0.4` (기존 유지)
  * High-Freq Gate (`> 200Hz`): `0.7` (기존 유지)
  * `F0 Median Filter Window`: `5` (플럭 스파이크 강제 평탄화)
* **Parsing Module:**
  * `MIN_DURATION_FRAMES`: `7` (70ms로 상향, 슬라이드 파편화 방지)
  * `TOLERANCE_FRAMES`: `5`

**[Results]**
1. **Noise Reduction:** 1차 실험의 노트 조기 끊김 및 파편화 현상은 유의미하게 감소함.
2. **Recall Drop (True Positive Loss):** 필터링이 과도하게 개입하여 정상적으로 연주된 짧은 노트들까지 전체적으로 증발함.
3. **Onset Smearing (Groove Loss):** 미디언 필터가 베이스의 어택(Attack) 특성을 훼손하여 전체적인 그루브가 무너짐. 하이 노트 솔로 파트에서 음의 시작과 끝 타이밍이 불명확해지는 부작용 발생.

**[Next Steps (Action Item)]**
* 베이스 어택 타이밍을 왜곡하는 F0 미디언 필터 전면 폐기 (1차 실험 베이스라인으로 롤백).
* 유실된 하이 노트 복구를 위해 High-Freq Threshold 하향 조정 (`0.7` -> `0.6`).
* 잃어버린 그루브(짧은 팝 노트) 복구를 위해 `MIN_DURATION_FRAMES` 하향 조정 (`7` -> `6`).

---

#### 🧪 Iteration 3: Groove Recovery (`midi_tunning_test3.mid`)

**[Parameters]**
* **Tracking Module (Dynamic Threshold Only):**
  * Low-Freq Gate (`< 80Hz`): `0.25`
  * Mid-Freq Gate (`80 ~ 200Hz`): `0.4`
  * High-Freq Gate (`> 200Hz`): `0.6` (하향 조정)
  * F0 Median Filter: **제거됨**
* **Parsing Module:**
  * `MIN_DURATION_FRAMES`: `6` (하향 조정)
  * `TOLERANCE_FRAMES`: `5`

**[Results]**
1. **Balance Restored:** 2차 실험에서 발생한 그루브 상실 및 어택 훼손 문제가 대다수 해결되며 밸런스를 찾음.
2. **Same-Pitch Retriggering Failure:** 근음 8비트/16비트 연타(동일 피치 반복) 구간에서 노트를 분할하지 못하고 하나의 긴 음(Legato)으로 뭉뚱그려 인식함. 파서가 주파수 변화에만 의존하여 발생하는 한계로 추정.
3. **Onset Latency:** 하이 노트 솔로 라인에서 원본 파형의 어택(Peak) 지점에 비해 MIDI 노트의 시작점(Onset)이 미세하게 뒤로 밀리는 딜레이 확인.
4. **High-frequency Ghost Notes:** 슬랩 파트에서 타격 잔향으로 인한 산발적인 초고역대 고스트 노트가 발생하여 정상적인 썸(Thumb) 트래킹을 방해함.

**[Next Steps (Action Item)]**
* **Onset-Aware Parsing:** 피치 변화 없이도 노트를 분할할 수 있도록, 트래킹 모듈에서 `librosa.onset.onset_strength`를 추출하여 파서(Parser)의 상태 머신에 강제 트리거(Re-trigger) 신호로 주입.
* **Latency Compensation:** CREPE 모델 윈도우링 특성상 발생하는 고정 지연(Latency)을 해소하기 위해 휴리스틱 기반의 시작점 후보정(예: -15ms) 로직 도입.

#### 🧪 Iteration 4: Onset-Aware Parser & Latency Compensation (`midi_tunning_test4.mid`)

**[Parameters]**
* **Tracking Module (Onset Data Injection):**
  * `librosa.onset.onset_detect`: 트래킹 단계에서 베이스 음원의 타격점(Onset) 포락선을 추출하여 `onset_mask`(Boolean 배열) 생성 후 파서로 전달.
* **Parsing Module (State Machine Upgrade):**
  * **Onset-aware Finalize & Restart**: 피치(Hz) 변화가 없더라도 `onset_mask`가 `True`인 프레임을 만나면 즉시 현재 노트를 종료하고 새로운 노트를 시작하는 동일 피치 연타(Retriggering) 강제 분할 알고리즘 신설.
  * `LATENCY_COMP_SEC`: `0.015` (15ms). CREPE 모델의 윈도우링 특성으로 인해 발생하는 고정 지연(Latency)을 해소하기 위해, 모든 추출된 노트의 시작점을 15ms 앞으로 당기는 휴리스틱 보정치 도입.

**[Results]**
1. **Same-Pitch Retriggering Failure:** 파서에 Onset 분할 로직을 도입했음에도 불구하고, 1파트의 근음 연타 구간이 시각적/청각적으로 여전히 쪼개지지 않음. 기본 Onset 탐지 감도(Threshold)가 낮아 저음역대의 부드러운 타격 에너지를 캡처하지 못한 것으로 진단됨.
2. **Latency Overcompensation:** 15ms의 고정 지연 보정이 베이스 어택 특성상 전체적인 노트 시작점이 원본 파형보다 미세하게 더 앞당겨지는(Rushed) 과보정 부작용 발생.

**[Next Steps (Action Item)]**
* **Sensitivity Maximization:** 저음역대 연타의 미세한 에너지 변화를 잡아내기 위해, Onset 탐지기의 감도 파라미터(`delta`)를 대폭 상향(수치 하향 조정)하고 타악기 간섭을 막기 위해 `fmax`를 400Hz로 제한.
* **Latency Adjustment:** 과보정된 타이밍을 교정하기 위해 `LATENCY_COMP_SEC`를 15ms에서 10ms(`0.010`)로 하향 조정.

#### 🧪 Iteration 5: Onset Sensitivity Maximization (`midi_tunning_test5.mid`)

**[Parameters]**
* **Tracking Module (Runner):**
  * `librosa.onset.onset_detect`: `delta` 파라미터를 기본값(0.07)에서 `0.03`으로 대폭 하향 조정하여 어택 탐지 감도(Sensitivity)를 한계치까지 극대화. 타악기 간섭을 줄이기 위해 `fmax`는 400Hz로 제한.
* **Parsing Module:**
  * `LATENCY_COMP_SEC`: `0.010` (10ms). 4차 실험의 15ms 과보정 현상을 해결하기 위해 하향 조정.

**[Results]**
1. **Same-Pitch Retriggering Failure (Feature Invariance):** 감도를 극대화했음에도 1파트의 저음역 근음 연타 구간은 분할되지 않음. 부드러운 베이스 연타는 진폭(Amplitude) 포락선만으로는 식별 가능한 물리적 특징점(Transient)이 부족함을 확인.
2. **False Polyphony (Sustain Regression):** Onset 감도가 너무 민감해진 결과, 베이스의 정상적인 서스테인(Sustain) 구간에서 발생하는 길게 유지되어야 할 음표들이 중간에 잘게 부서지는(Fragmentation) 치명적인 현상 발생.

**[Next Steps (Action Item)]**
* **Rollback Onset Sensitivity:** 5차 실험의 과도한 감도를 폐기하고 `delta`를 0.07(기본값)로 롤백하여 서스테인 노트를 보호.
* **Confidence-Aware Retriggering (Test 6):** 진폭(Amplitude) 기반 분할을 포기하고, 모델의 예측 신뢰도(Confidence) 하락을 Feature로 활용. 피치가 유지되더라도 Confidence가 순간적으로 특정 임계값(`0.3`) 이하로 떨어졌다 회복되면 재타현(Retrigger)으로 간주하여 노트를 분할하는 로직 도입.

#### 🧪 Iteration 6: Confidence-Aware Retriggering (`midi_tunning_test6.mid`)

**[Parameters]**
* **Tracking Module (Runner Rollback):**
  * `librosa.onset.onset_detect`: 5차 실험의 과도한 감도를 폐기하고 `delta`를 `0.07`(기본값)로 롤백. 서스테인 파편화(False Polyphony) 원천 차단.
* **Parsing Module (Feature Shift):**
  * `RETRIGGER_CONF_THRESH`: `0.3` (신설). 진폭(Amplitude) 대신 모델의 예측 신뢰도(Confidence)를 Feature로 활용. 피치가 유지되더라도 Confidence가 0.3 미만으로 순간 하락하면 베이시스트의 재타현(Retrigger)에 의한 배음 간섭으로 간주하여 노트를 강제 분할함.
  * `LATENCY_COMP_SEC`: `0.010` (10ms 유지).

**[Results]**
1. **Same-Pitch Splitting Success:** 1파트의 저음역 근음 연타 구간이 드디어 박자에 맞게 분할(Retriggering)되기 시작함. 부드러운 연타 분할에는 진폭보다 '신뢰도의 순간적 균열'이 훨씬 유효한 지표임을 증명.
2. **Sustain Truncation Bug:** 파서의 상태 머신 설계 결함 발견. 신뢰도 하락 시 이전 노트를 종료한 후, 대기(Blank) 상태로 가지 않고 곧바로 새 노트를 시작해버려 정상적인 서스테인(Sustain)의 꼬리 부분이 짧게 깎여나가는 부작용 발생.
3. **Slap Octave Jumps:** 코랩 기반의 빠른 테스트 환경(Test Harness)을 구성하는 과정에서, 기존 파이프라인(Phase 2)에 존재하던 `clean_octave_errors_smart` 모듈을 누락하여 슬랩 주법 특유의 강한 배음이 옥타브 점프 에러로 발현됨.

**[Next Steps (Action Item)]**
* **State Machine Bug Fix:** 파서 로직을 수정하여, 신뢰도 하락으로 노트를 종료한 직후에는 `current_note = None`으로 전환해 노이즈 구간 동안 대기하도록 조치 (서스테인 보호).
* **Octave Correction Restoration:** 트래킹 모듈에 스마트 옥타브 보정 함수를 복구하여 슬랩 파트의 옥타브 튐 현상 억제.
* **Threshold Fine-tuning:** 서스테인 꼬리를 더 길게 살리기 위해 `RETRIGGER_CONF_THRESH`를 `0.2`로 하향.

---

#### 🧪 Iteration 7: Sustain Protection & Octave Correction (`midi_tunning_test7.mid`)

**[Parameters]**
* **Tracking Module:**
  * `clean_octave_errors_smart`: 배음 오인 방어 모듈 복구. Onset 마스크를 옥타브 보정기에도 주입하도록 Runner의 실행 순서 재조정 (Onset Mask 생성 -> CREPE 추론 & 옥타브 보정 -> Parser 파싱).
* **Parsing Module (Bug Fix):**
  * `RETRIGGER_CONF_THRESH`: `0.2` (0.3에서 하향. 서스테인 보호력 강화).
  * **State Machine Transition Fix**: Confidence 하락(`< 0.2`) 조건 충족 시, 기존